<a href="https://colab.research.google.com/github/cirsam/fine-tune-AI-model-with-unsloth-studio/blob/main/%F0%9F%92%A7_LFM2_5_Inference_with_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💧 LFM2.5 Inference with transformers

This notebook allows you to easily run LFM2 and LFM2.5 models (like [`LiquidAI/LFM2.5-1.2B-Instruct`](https://huggingface.co/LiquidAI/LFM2.5-1.2B-Instruct)) with Hugging Face's [transformers](https://github.com/huggingface/transformers) library.

You can run it on GPU or CPU by switching the runtime (`Runtime` → `Change runtime type`).

In [2]:
!pip install -qqqU transformers --progress-bar off

from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

# Load model and tokenizer
model_id = "LiquidAI/LFM2.5-1.2B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="bfloat16",
#   attn_implementation="flash_attention_2" <- uncomment on compatible GPU
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.78k [00:00<?, ?B/s]

In [3]:
import torch

# Generate answer
prompt = "What is C. elegans?"

# The tokenizer.apply_chat_template with return_tensors="pt" typically returns a BatchEncoding object.
# This object is a dictionary-like structure containing 'input_ids' and 'attention_mask' tensors.
encoded_input = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    # The 'tokenize' argument is usually not necessary here as apply_chat_template handles tokenization
    # and structuring for chat templates. It can be removed if not explicitly needed.
)

# Extract input_ids and attention_mask and move them to the model's device
input_ids = encoded_input["input_ids"].to(model.device)
attention_mask = encoded_input["attention_mask"].to(model.device)

output = model.generate(
    input_ids,
    attention_mask=attention_mask, # Pass the attention_mask
    do_sample=True,
    temperature=0.1,
    top_k=50,
    top_p=0.1,
    repetition_penalty=1.05,
    max_new_tokens=512,
    streamer=streamer,
)

**C. elegans** (pronounced "see-uh-nahs") is a small, transparent nematode worm commonly used in biological research. Here's a detailed overview:

### 1. **Basic Facts**
- **Scientific Name:** *Caenorhabditis elegans*
- **Type:** Nematode (roundworm)
- **Size:** About 1–2 mm long
- **Color:** Pale brown or white
- **Habitat:** Found in soil and decaying organic matter

### 2. **Why It's Important**
- **Model Organism:** Widely used in molecular biology, genetics, developmental biology, and neuroscience.
- **Short Lifespan:** Lives only about 2–3 weeks, making it ideal for studying life processes over short time frames.
- **Genome:** One of the smallest known genomes (about 20,000 genes), which makes it easier to study gene function.

### 3. **Key Features**
- **Transparent Body:** Allows researchers to observe internal structures under a microscope.
- **Well-Characterized Genome:** Facilitates genetic studies.
- **Simple Nervous System:** Has only 302 neurons, making it easy to map neu